# [제출용 안내] 최종 제출 코드 검증 기록

이 폴더(mission2)의 **최종 코드**(코드 정리 + 추론 속도 개선 후)를 Colab A100 에서 풀어, 대회 채점 명령으로 공식 Validation 3,640통화 전체를 돌린 기록이다.
아래 셀들의 코드와 출력은 실행한 그대로이며(새 런타임에서 1~7번을 차례로 한 번 실행, 오류 없음), 이 안내 셀만 제출할 때 맨 위에 덧붙였다.

- 결과: 정확도 92.8957% (103,994 / 111,947), 대원 recall 88.36% / 신고자 recall 97.10%, 추론 12.1분(발화당 6.5ms, 전처리 작업자 8개),
  `requirements.txt` 만 설치(Python 3.13.15), 인터넷 차단(`HF_HUB_OFFLINE=1`), 체크포인트 sha256 일치.
- 코드는 최종 코드만 담은 압축 파일에서, 가중치(1.27GB)는 앞선 검증 때 올린 압축 파일에서 꺼내 썼다(체크포인트 sha256 으로 같은 파일임을 확인).
- 속도 개선 전 코드(추론 20.2분)로 같은 방식으로 돌린 결과와 111,947발화 모두 판정이 같다(6번 셀 마지막 줄). 그 결과는 다시 코드 정리 전 결과와도 모두 같았다.

# Mission 2 최종 제출물 재검증 v3 (추론 속도 개선 후, Colab GPU, 공식 Validation 전체)

코드 정리(불필요한 코드 삭제, 한글 주석) 후의 **제출 폴더 코드**를 풀고, 가중치는 지난 검증 때 올린 압축 파일에서 꺼내서, 대회 채점 명령(`python inference.py ...`)으로 공식 Validation 3,640통화 전체를 돌린다.
- 확인하는 것: (1) 인터넷 없이 실행되는지, (2) 출력 형식, (3) 정확도가 92.89% 근처로 재현되는지, (4) GPU 추론 시간.
- 정답은 라벨 JSON 의 `speaker` 로 이 노트북에서만 채점한다(제출 코드는 `speaker` 를 읽지 않는다).

**사용법**: GPU 런타임(A100/L4/T4 아무거나)에서 "모두 실행". 끝나면 출력이 보이는 상태로 저장(Ctrl+S).

In [1]:
# 1. 설정
DRIVE_ROOT = "/content/drive/MyDrive/데이터 크리에이터 캠프"
SUBMISSION_ZIP = f"{DRIVE_ROOT}/제출_C/m2_final_code_v3.zip"    # 속도 개선한 코드만 담은 압축 파일 (가중치 없음)
WEIGHTS_ZIP = f"{DRIVE_ROOT}/제출_C/m2_final_check.zip"         # 지난 검증 때 올린 압축 파일 (가중치를 여기서 꺼냄)
PREVIOUS_RESULT = f"{DRIVE_ROOT}/제출_C/검증결과_v2/mission2.csv"   # 속도 개선 전(v2) 코드의 공식 Validation 결과 (발화별 비교용)
ZIP_DIR = f"{DRIVE_ROOT}/대학부 데이터"                           # 대회 원본 zip
RESULT_DIR = f"{DRIVE_ROOT}/제출_C/검증결과_v3"
EXPECTED_ACCURACY = 92.8957  # 속도 개선 전(v2) 코드의 값
EXPECTED_CKPT_SHA256 = "35e5e586d33a8f7ab5d0bf174ecbba33cf8de152395b9b97ea31a99bb73cac94"

In [2]:
# 2. Drive 연결, GPU 확인, 제출 폴더 풀기
import glob, hashlib, json, os, shutil, subprocess, time
from google.colab import drive
drive.mount("/content/drive")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)
assert os.path.isfile(SUBMISSION_ZIP), f"없음: {SUBMISSION_ZIP}"
shutil.rmtree("/content/sub", ignore_errors=True)
os.makedirs("/content/sub")
shutil.copyfile(SUBMISSION_ZIP, "/content/sub.zip")
subprocess.run("cd /content/sub && unzip -q /content/sub.zip", shell=True, check=True)
SUB = "/content/sub/mission2"
os.makedirs(f"{SUB}/checkpoints", exist_ok=True)
subprocess.run(f"unzip -q -j '{WEIGHTS_ZIP}' mission2/checkpoints/m2_final_ensemble.pt -d {SUB}/checkpoints", shell=True, check=True)
print(sorted(os.listdir(SUB)))

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

CKPT = f"{SUB}/checkpoints/m2_final_ensemble.pt"
ckpt_sha = sha256(CKPT)
print("체크포인트 sha256:", ckpt_sha, "-> 기대값과", "같음" if ckpt_sha == EXPECTED_CKPT_SHA256 else "다름!!")
assert ckpt_sha == EXPECTED_CKPT_SHA256

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB

['README.md', 'SOURCE_COMMIT.txt', 'checkpoints', 'inference.py', 'requirements.txt', 'scripts', 'src', 'train_mission2_l2_from_raw.ipynb', 'train_mission2_wavlm_seed1.ipynb', 'train_mission2_wavlm_seed2.ipynb', 'verify_mission2_final_on_gpu.ipynb']
체크포인트 sha256: 35e5e586d33a8f7ab5d0bf174ecbba33cf8de152395b9b97ea31a99bb73cac94 -> 기대값과 같음


In [3]:
# 3. requirements.txt 설치 (채점 환경처럼 이 파일만으로) + 설치된 버전 기록
subprocess.run(f"pip install -q -r {SUB}/requirements.txt", shell=True, check=True)
print(subprocess.run("python --version; pip list 2>/dev/null | grep -i -E '^(numpy|pandas|librosa|torch|transformers|scikit-learn|lightgbm) '",
                     shell=True, capture_output=True, text=True).stdout)

Python 3.13.15
librosa                               0.11.0
lightgbm                              4.6.0
numpy                                 2.1.3
pandas                                2.2.3
scikit-learn                          1.6.1
torch                                 2.11.0+cu130
transformers                          5.19.0



In [4]:
# 4. 공식 Validation 풀기 + 한 폴더로 모으기. 예전 학습 노트북과 같은 "전부 풀기"(--n-train-calls 0, 약 15분):
#    일부만 뽑는 방식은 소리와 라벨이 다른 zip 조각에 나뉜 통화를 빠뜨린다(이 노트북의 첫 실행에서 3,640통화 중 1,541개만 풀림)
subprocess.run(f"python {SUB}/scripts/extract_dcc_sample_from_zips.py --zip-dir '{ZIP_DIR}' --output-dir /content/val_data "
               "--n-train-calls 0", shell=True, check=True)
VAL = "/content/val_flat"
shutil.rmtree(VAL, ignore_errors=True)
os.makedirs(VAL)
found = {}
for path in glob.glob("/content/val_data/**/*", recursive=True):
    if os.path.isfile(path) and path.endswith((".wav", ".json")) and "/Validation/" in path:
        found.setdefault(os.path.basename(path), []).append(path)
print("Validation 이 풀린 폴더들:", sorted({p.split("/Validation/")[0] + "/Validation" for paths in found.values() for p in paths}))
duplicated = {name: paths for name, paths in found.items() if len(paths) > 1}
print("이름이 겹치는 파일:", len(duplicated))
for name, paths in found.items():
    os.symlink(paths[0], os.path.join(VAL, name))
n_wav = len(glob.glob(f"{VAL}/*.wav")); n_json = len(glob.glob(f"{VAL}/*.json"))
print("모은 폴더:", VAL, "| wav", n_wav, "json", n_json)
assert n_wav == n_json == 3640, "공식 Validation 은 3,640통화여야 한다"

Validation 이 풀린 폴더들: ['/content/val_data/Validation']
이름이 겹치는 파일: 0
모은 폴더: /content/val_flat | wav 3640 json 3640


In [5]:
# 5. 채점 명령 그대로 실행 (인터넷 차단 상태로: Hugging Face 접속을 막는다)
os.makedirs("/content/out", exist_ok=True)
env = {**os.environ, "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"}
cmd = (f"cd {SUB} && python inference.py --audio_dir '{VAL}' --label_dir '{VAL}' "
       f"--ckpt_path checkpoints/m2_final_ensemble.pt --output /content/out/mission2.csv "
       f"--probs_output /content/out/mission2_probs.csv")
print("$", cmd)
started = time.time()
process = subprocess.Popen(cmd, shell=True, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
assert process.wait() == 0, "실행 실패"
WALL = time.time() - started
print(f"전체 소요 {WALL / 60:.1f}분 (모델 로딩 포함)")

$ cd /content/sub/mission2 && python inference.py --audio_dir '/content/val_flat' --label_dir '/content/val_flat' --ckpt_path checkpoints/m2_final_ensemble.pt --output /content/out/mission2.csv --probs_output /content/out/mission2_probs.csv
device=cuda (NVIDIA A100-SXM4-40GB), WavLM batch_size=64, preprocessing workers=8, CNN members=3, WavLM members=2, weights cnn=0.4 wavlm=[0.3, 0.3]
  200/3640 calls, 6193 utterances, 1.1 min
  400/3640 calls, 12280 utterances, 1.7 min
  600/3640 calls, 19294 utterances, 2.4 min
  800/3640 calls, 25430 utterances, 3.1 min
  1000/3640 calls, 31363 utterances, 3.7 min
  1200/3640 calls, 37066 utterances, 4.3 min
  1400/3640 calls, 42923 utterances, 4.9 min
  1600/3640 calls, 48851 utterances, 5.6 min
  1800/3640 calls, 55013 utterances, 6.2 min
  2000/3640 calls, 60934 utterances, 6.8 min
  2200/3640 calls, 67052 utterances, 7.4 min
  2400/3640 calls, 73084 utterances, 8.1 min
  2600/3640 calls, 79195 utterances, 8.7 min
  2800/3640 calls, 84940 uttera

In [6]:
# 6. 채점 (라벨 JSON 의 speaker) + 출력 형식 확인
import pandas as pd
out = pd.read_csv("/content/out/mission2.csv")
print("첫 줄(컬럼):", list(out.columns), "| 행 수:", len(out), "| speaker 값:", sorted(out["speaker"].unique().tolist()))
assert list(out.columns) == ["audio file name", "startAt", "endAt", "speaker"]

truth = []
for json_path in sorted(glob.glob(f"{VAL}/**/*.json", recursive=True), key=lambda p: os.path.splitext(os.path.basename(p))[0]):
    stem = os.path.splitext(os.path.basename(json_path))[0]
    for utt in json.load(open(json_path, encoding="utf-8"))["utterances"]:
        truth.append((stem + ".wav", utt["startAt"], utt["endAt"], int(utt["speaker"])))
truth = pd.DataFrame(truth, columns=["audio file name", "startAt", "endAt", "truth"])
assert len(truth) == len(out)
assert (truth["audio file name"].values == out["audio file name"].values).all(), "행 순서가 라벨과 다르다"
assert (truth["startAt"].values == out["startAt"].values).all() and (truth["endAt"].values == out["endAt"].values).all()
correct = out["speaker"].values == truth["truth"].values
accuracy = correct.mean() * 100
recall0 = (out["speaker"].values[truth["truth"].values == 0] == 0).mean() * 100
recall1 = (out["speaker"].values[truth["truth"].values == 1] == 1).mean() * 100
print(f"공식 Validation 정확도 {accuracy:.4f}% ({int(correct.sum())}/{len(out)}), 대원 recall {recall0:.2f}% / 신고자 recall {recall1:.2f}%")
print(f"기대값 {EXPECTED_ACCURACY}% 와의 차이 {accuracy - EXPECTED_ACCURACY:+.4f}%p")
previous = pd.read_csv(PREVIOUS_RESULT)
assert list(previous.columns) == list(out.columns) and len(previous) == len(out)
same_rows = (previous[["audio file name", "startAt", "endAt"]].values == out[["audio file name", "startAt", "endAt"]].values).all()
changed = int((previous["speaker"].values != out["speaker"].values).sum())
print(f"속도 개선 전(v2) 결과와 비교: 행 순서 같음={same_rows}, 판정이 다른 발화 {changed}개 / {len(out)}")

첫 줄(컬럼): ['audio file name', 'startAt', 'endAt', 'speaker'] | 행 수: 111947 | speaker 값: [0, 1]
공식 Validation 정확도 92.8957% (103994/111947), 대원 recall 88.36% / 신고자 recall 97.10%
기대값 92.8957% 와의 차이 +0.0000%p
속도 개선 전(v2) 결과와 비교: 행 순서 같음=True, 판정이 다른 발화 0개 / 111947


In [7]:
# 7. 결과를 Drive 에 저장 (PC 에서 연구용 확률 파일과 발화별로 대조한다)
os.makedirs(RESULT_DIR, exist_ok=True)
for name in ("mission2.csv", "mission2_probs.csv"):
    shutil.copyfile(f"/content/out/{name}", f"{RESULT_DIR}/{name}")
summary = {"accuracy": accuracy, "recall_0": recall0, "recall_1": recall1, "rows": len(out), "wall_minutes": WALL / 60,
           "ckpt_sha256": ckpt_sha, "gpu": subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True,
                                                          capture_output=True, text=True).stdout.strip()}
json.dump(summary, open(f"{RESULT_DIR}/summary.json", "w"), ensure_ascii=False, indent=1)
print(summary)

{'accuracy': np.float64(92.89574530804757), 'recall_0': np.float64(88.36103630792088), 'recall_1': np.float64(97.09820660218237), 'rows': 111947, 'wall_minutes': 12.70528211593628, 'ckpt_sha256': '35e5e586d33a8f7ab5d0bf174ecbba33cf8de152395b9b97ea31a99bb73cac94', 'gpu': 'NVIDIA A100-SXM4-40GB'}
